# 03 — Visualizing matrices and linear transformations

**Goal:** use heatmaps and 2D geometry to see how matrix values relate to the transformations they represent.

A 2 × 2 matrix maps a point x in the plane to Ax. Its columns are the images of the standard basis vectors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matrixlib import Matrix

np.set_printoptions(precision=3, suppress=True)

## 1. Plot a matrix heatmap

In [ ]:
A = np.array([[1.0, -2.0], [3.0, 0.5]])
lim = np.max(np.abs(A))
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(A, cmap="coolwarm", vmin=-lim, vmax=lim)
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        ax.text(j, i, f"{A[i,j]:g}", ha="center", va="center")
ax.set_title("A matrix is also a small data grid")
fig.colorbar(im, ax=ax)
plt.show()

## 2. Draw vectors and a coordinate grid

The transformation is applied to every point. We use a square grid as a visual probe of the mapping.

In [ ]:
def plot_grid(ax, matrix=None, title="Original grid", extent=3):
    """Plot a coordinate grid, optionally transformed by a 2x2 matrix."""
    points = []
    for x in range(-extent, extent + 1):
        points.append(np.array([[x, -extent], [x, extent]], dtype=float))
    for y in range(-extent, extent + 1):
        points.append(np.array([[-extent, y], [extent, y]], dtype=float))
    for segment in points:
        mapped = segment if matrix is None else (matrix @ segment.T).T
        ax.plot(mapped[:, 0], mapped[:, 1], color="steelblue", alpha=0.45, linewidth=1)
    basis = np.eye(2) if matrix is None else matrix @ np.eye(2)
    ax.quiver([0, 0], [0, 0], basis[0, :], basis[1, :],
              angles="xy", scale_units="xy", scale=1, color=["darkorange", "seagreen"])
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvline(0, color="black", linewidth=0.6)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.grid(alpha=0.2)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_grid(axes[0], title="Original")
rotation = np.array([[0, -1], [1, 0]], dtype=float)
plot_grid(axes[1], rotation, "90 degree rotation")
plt.tight_layout()
plt.show()

## 3. Transformation presets

Common 2D linear transformations are represented by 2 × 2 matrices. Angles in the example are in radians.

In [ ]:
theta = np.deg2rad(30)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
scale = np.array([[2.0, 0.0], [0.0, 0.5]])
reflection = np.array([[1.0, 0.0], [0.0, -1.0]])
shear = np.array([[1.0, 1.0], [0.0, 1.0]])

presets = {"rotation 30°": R, "nonuniform scale": scale,
           "reflection": reflection, "x shear": shear}
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (name, matrix) in zip(axes, presets.items()):
    plot_grid(ax, matrix, name, extent=2)
plt.tight_layout()
plt.show()

## 4. Determinant and area

For a 2D linear transformation, the absolute determinant is the area-scaling factor. A negative determinant reverses orientation; a zero determinant collapses the plane into a lower-dimensional set.

In [ ]:
for name, matrix in presets.items():
    determinant = np.linalg.det(matrix)
    print(f"{name:18s} det={determinant: .4f}; area scale={abs(determinant):.4f}")

singular = np.array([[1.0, 2.0], [2.0, 4.0]])
print("singular matrix determinant:", np.linalg.det(singular))
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
plot_grid(axes[0], scale, "Invertible: area changes", extent=2)
plot_grid(axes[1], singular, "Singular: plane collapses", extent=2)
plt.tight_layout()
plt.show()

## 5. Use the library for numerical properties

The plots above use NumPy directly because Matplotlib needs coordinate arrays. The Matrix wrapper can compute the same matrix properties and provides validation.

In [ ]:
for name, array in presets.items():
    matrix = Matrix(array)
    print(name)
    print("matrix:\n", matrix)
    print("determinant:", matrix.determinant())
    print("rank:", matrix.rank())
    print("condition number:", matrix.condition_number())
    print()

## Exercises

1. Add a 45° rotation and plot it.
2. Choose a matrix with determinant -1 and interpret its effect.
3. Change the shear coefficient and observe the grid.
4. Construct a singular 2 × 2 matrix and explain which dimension collapses.
5. Apply two transformations in sequence. Compare R @ S with S @ R to observe that order generally matters.
6. Transform the four corners of a unit square and estimate the resulting area from the determinant.